# Tugas 1 — Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Target Check-in:** Pertemuan 8 (Mulai dikerjakan Pertemuan 6)  
**Nama Mahasiswa:** Ayesha Fahrelia Ningrum  
**NIM:** 202310370311277
**Username GitHub:** FahreliaYesha

---

## Panduan Pengerjaan Milestone 3:
1. **Sumber Data:** Gunakan dataset yang telah dibersihkan pada Milestone 2 (`02_data_cleaning.ipynb`).
2. **Analisis Temporal / Windowing:** Terapkan minimal satu operasi temporal (Polars `group_by_dynamic()`, `rolling_mean()`, `shift()`, atau DuckDB window function `OVER (...)`).
3. **Visualisasi Interaktif:** Bangun minimal 6 visualisasi interaktif informatif menggunakan Plotly Express / Graph Objects atau Altair.
4. **Perumusan Insight:** Tuliskan minimal 5 temuan atau wawasan analitik mendalam berdasarkan hasil visualisasi data di atas.


## Bagian 1: Inisialisasi dan Memuat Data Bersih

Muat dataset hasil pembersihan dari folder `data/` menggunakan Polars.

In [1]:

from pathlib import Path
import re

import polars as pl
import duckdb
import pandas as pd
import plotly.express as px

# ============================================================
# 1. MENENTUKAN ROOT REPOSITORY DAN PATH DATASET
# ============================================================

# Notebook berada di /home/jovyan/work/notebooks
# Root repository berada di /home/jovyan/work
ROOT_DIR = Path.cwd().parent

DATA_PATH = (
    ROOT_DIR
    / "data"
    / "raw"
    / "indonesia_palm_oil_v1_2_5.csv"
)

OUTPUT_DIR = ROOT_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Root repository :", ROOT_DIR)
print("Lokasi dataset  :", DATA_PATH)
print("Folder output   :", OUTPUT_DIR)

# ============================================================
# 2. VALIDASI DATASET
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}\n"
        "Pastikan file CSV tersedia di folder "
        "data/raw pada root repository."
    )

# ============================================================
# 3. MEMBACA DATA DENGAN POLARS LAZY EVALUATION
# ============================================================

lf = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10000,
    ignore_errors=True,
    truncate_ragged_lines=True
)

print("\nSchema dataset:")
print(lf.collect_schema())

# ============================================================
# 4. MEMUAT DATASET
# ============================================================

df = lf.collect()

print("\nJumlah baris :", df.height)
print("Jumlah kolom :", df.width)

print("\nLima baris pertama dataset:")
display(df.head())


Root repository : /home/jovyan/work
Lokasi dataset  : /home/jovyan/work/data/raw/indonesia_palm_oil_v1_2_5.csv
Folder output   : /home/jovyan/work/data/processed

Schema dataset:
Schema({'year': Int64, 'country_of_production': String, 'product_type': String, 'forest_500_palm_oil': String, 'zero_deforestation_indonesia_palm_oil': String, 'province_of_production': String, 'kabupaten_of_production': String, 'mill': String, 'mill_trase_id': String, 'mill_group': String, 'refinery': String, 'refinery_trase_id': String, 'refinery_group': String, 'exporter': String, 'exporter_group': String, 'port_of_export': String, 'importer': String, 'importer_group': String, 'country_of_first_import': String, 'economic_bloc': String, 'province_of_production_trase_id': String, 'kabupaten_of_production_trase_id': String, 'country_of_production_trase_id': String, 'country_of_first_import_trase_id': String, 'palm_oil_deforestation_10_year_total_exposure': String, 'palm_oil_deforestation_annual_exposure': Stri

year,country_of_production,product_type,forest_500_palm_oil,zero_deforestation_indonesia_palm_oil,province_of_production,kabupaten_of_production,mill,mill_trase_id,mill_group,refinery,refinery_trase_id,refinery_group,exporter,exporter_group,port_of_export,importer,importer_group,country_of_first_import,economic_bloc,province_of_production_trase_id,kabupaten_of_production_trase_id,country_of_production_trase_id,country_of_first_import_trase_id,palm_oil_deforestation_10_year_total_exposure,palm_oil_deforestation_annual_exposure,total_emission_exposure,net_emission_luc_exposure,emission_subsidence_exposure,gross_emission_luc_exposure,emissions_fire_on_peat_exposure,volume,oil_palm_ha,fob
i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ACCESS BALI""","""ACCESS BALI""","""TANJUNG PRIOK""","""JOV-HENTON LTD""","""JOV-HENTON LTD""","""NIGERIA""","""NIGERIA""","""ID-XX""","""ID-XXXX""","""ID""","""NG""",null,null,null,null,null,null,null,22.8,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""ACALPO""","""ACALPO""","""INDIA""","""INDIA""","""ID-XX""","""ID-XXXX""","""ID""","""IN""",null,null,null,null,null,null,null,3000.0,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""BUNGE""","""BUNGE""","""INDIA""","""INDIA""","""ID-XX""","""ID-XXXX""","""ID""","""IN""",null,null,null,null,null,null,null,1000.0,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""CARGILL""","""CARGILL""","""INDIA""","""INDIA""","""ID-XX""","""ID-XXXX""","""ID""","""IN""",null,null,null,null,null,null,null,1000.0,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""CARGILL""","""CARGILL""","""TANZANIA""","""TANZANIA""","""ID-XX""","""ID-XXXX""","""ID""","""TZ""",null,null,null,null,null,null,null,500.0,null,null


## Bagian 2: Analisis Temporal dan Window Functions (Materi Pertemuan 6)

Terapkan teknik windowing Polars (`group_by_dynamic`, `rolling_mean`, `shift`, atau DuckDB window functions)
untuk menangkap tren waktu, pola musiman, atau pergerakan akumulatif.

In [2]:
# Ringkasan tipe data
display(pd.DataFrame({
    "kolom": df.columns,
    "tipe_data": [str(dtype) for dtype in df.dtypes],
    "jumlah_missing": [df[c].null_count() for c in df.columns],
    "persentase_missing": [
        round(df[c].null_count() / max(df.height, 1) * 100, 2)
        for c in df.columns
    ]
}))

print("Duplikasi penuh:", int(df.is_duplicated().sum()))

# DuckDB SUMMARIZE
con = duckdb.connect(":memory:")
con.register("dataset", df.to_pandas())

display(con.execute("SUMMARIZE SELECT * FROM dataset").df())

,kolom,tipe_data,jumlah_missing,persentase_missing
0,year,Int64,0,0.00
1,country_of_production,String,0,0.00
2,product_type,String,0,0.00
3,forest_500_palm_oil,String,0,0.00
4,zero_deforestation_indonesia_palm_oil,String,0,0.00
5,province_of_production,String,0,0.00
6,kabupaten_of_production,String,0,0.00
7,mill,String,0,0.00
8,mill_trase_id,String,0,0.00
9,mill_group,String,0,0.00


Duplikasi penuh: 0


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,year,BIGINT,2013,2022,10,2020.0519167128343,1.3540499690565622,2019,2020,2021,1694907,0.00
1,country_of_production,VARCHAR,INDONESIA,INDONESIA,1,NaN,NaN,NaN,NaN,NaN,1694907,0.00
2,product_type,VARCHAR,PALM OIL,REFINED PALM OIL,2,NaN,NaN,NaN,NaN,NaN,1694907,0.00
3,forest_500_palm_oil,VARCHAR,0,NOT ASSESSED,7,NaN,NaN,NaN,NaN,NaN,1694907,0.00
4,zero_deforestation_indonesia_palm_oil,VARCHAR,COMPANY COMMITMENT,UNKNOWN,4,NaN,NaN,NaN,NaN,NaN,1694907,0.00
5,province_of_production,VARCHAR,ACEH,UNKNOWN,30,NaN,NaN,NaN,NaN,NaN,1694907,0.00
6,kabupaten_of_production,VARCHAR,ACEH BARAT,WAY KANAN,282,NaN,NaN,NaN,NaN,NaN,1694907,0.00
7,mill,VARCHAR,ABDI BORNEO PLANTATIONS (ABDI),YUNI BERSAUDARA SEJAHTERA,1231,NaN,NaN,NaN,NaN,NaN,1694907,0.00
8,mill_trase_id,VARCHAR,ID-PALM-MILL-00001,ID-PALM-MILL-X,1232,NaN,NaN,NaN,NaN,NaN,1694907,0.00
9,mill_group,VARCHAR,ADIMULIA,ZONERGY,204,NaN,NaN,NaN,NaN,NaN,1694907,0.00


In [3]:
def normalisasi(nama):
    return re.sub(r"[^a-z0-9]+", "_", nama.lower()).strip("_")

nama_normal = {c: normalisasi(c) for c in df.columns}

def cari_kolom(kandidat):
    for kata in kandidat:
        for kolom, nama in nama_normal.items():
            if nama == kata:
                return kolom

    for kata in kandidat:
        for kolom, nama in nama_normal.items():
            if kata in nama:
                return kolom

    return None

kolom_tahun = cari_kolom(["year", "reporting_year"])
kolom_wilayah = cari_kolom([
    "province", "provinsi", "district",
    "kabupaten", "region", "regency"
])
kolom_tujuan = cari_kolom([
    "destination_country", "destination",
    "import_country", "country"
])

kolom_numerik = [
    c for c, dtype in zip(df.columns, df.dtypes)
    if dtype.is_numeric()
]

kolom_volume = cari_kolom([
    "volume_tonnes", "volume_tons", "volume",
    "quantity", "production_volume", "tonnes"
])

if kolom_volume not in kolom_numerik:
    kolom_volume = None

print("Kolom tahun:", kolom_tahun)
print("Kolom wilayah:", kolom_wilayah)
print("Kolom tujuan:", kolom_tujuan)
print("Kolom numerik:", kolom_numerik)
print("Kolom volume:", kolom_volume)

Kolom tahun: year
Kolom wilayah: province_of_production
Kolom tujuan: country_of_production
Kolom numerik: ['year', 'volume']
Kolom volume: volume


## Bagian 3: Minimal 6 Visualisasi Interaktif (Plotly / Altair)

Bangun minimal 6 visualisasi interaktif yang mencakup:
- Visualisasi 1: Tren deret waktu (Line chart dengan range slider)
- Visualisasi 2: Distribusi variabel utama (Box plot / Histogram / Violin plot)
- Visualisasi 3: Korelasi antar-variabel numerik (Heatmap / Scatter plot sampling)
- Visualisasi 4: Komposisi kategori utama (Bar chart / Treemap)
- Visualisasi 5: Analisis siklus musiman / perbandingan kelompok
- Visualisasi 6: Pola spasial atau korelasi multi-dimensi

In [4]:
#Visualisasi 1 Tren

In [5]:
#Visualisasi 2 Distribusi

In [6]:
#Visualisasi 3 Korelasi

In [7]:
#Visualisasi 4Barchart Komposisi

In [8]:
#Visualisasi 5 Musiman/Temporal

In [9]:
#Visualisasi 6 Pola Spasial

## Bagian 4: Rumusan 5 Insight Analitik Utama

Tuliskan minimal 5 temuan atau wawasan penting yang Anda peroleh dari visualisasi di atas:

1. **Insight 1 (Tren Waktu & Musiman):**  
   [Jelaskan temuan terkait tren jangka panjang atau pola berulang yang teridentifikasi]

2. **Insight 2 (Distribusi & Anomali):**  
   [Jelaskan temuan terkait sebaran data, ketimpangan, atau keberadaan pencilan penting]

3. **Insight 3 (Korelasi Kunci):**  
   [Jelaskan hubungan signifikan antar-variabel yang dapat dijadikan dasar keputusan bisnis/kebijakan]

4. **Insight 4 (Perilaku Kategori / Segmen):**  
   [Jelaskan perbedaan karakteristik performa antar kelompok/wilayah/kategori produk]

5. **Insight 5 (Rekomendasi Praktis):**  
   [Jelaskan tindakan nyata atau strategi operasional yang direkomendasikan berdasarkan seluruh temuan di atas]
